# Inference

This notebook validates the saved next-category recommendation models and defines a reusable inference function.

> **Hypothesis test: anonymisation cost (classification).** Release: `regions_anonymised` (regions release). Feature set: `03_pure_record` (12 predictors). Code is unchanged from the official `04_fm_inference.ipynb` except for paths, the feature-set definition, the holdout split (loaded from the shared `split_assignment.csv`), and the validation against the package module, which is replaced by a reproduction check of this run's saved models.

The function returns the primary recommended category together with a ranked list of alternative categories and their predicted probabilities.

In [1]:
# -------------------------------------------------------------------
# Load Final Models And Define Inference Function
# -------------------------------------------------------------------

from pathlib import Path

import joblib
import pandas as pd

MODEL_PREFIX = "regions_pure_record"

MODELS_DIR = Path("../../../../models/hypothesis_testing")

# DROPPED_FEATURES (predictors removed in this feature set) is defined in the
# section 'Recreate Holdout Test Split', once the modelling table is loaded.

top1_model = joblib.load(
    MODELS_DIR / f"{MODEL_PREFIX}_top1.joblib"
)

ranking_model = joblib.load(
    MODELS_DIR / f"{MODEL_PREFIX}_ranking.joblib"
)


def predict_next_categories(feature_row: pd.DataFrame) -> dict:
    """
    Predict the next recommended household categories.

    Parameters
    ----------
    feature_row : pd.DataFrame
        A single-row DataFrame containing the same model features
        used during training.

    Returns
    -------
    dict
        Top-1 recommendation plus ranked Top-2 and Top-3 results.
    """

    # Remove identifiers not used by the trained models.
    model_input = feature_row.drop(
        columns=[
            "Property_ID",
            "Customer_ID",
            "Prediction_Time",
        ] + DROPPED_FEATURES,
        errors="ignore",
    ).copy()

    # -----------------------------------------------------------
    # Top-1 model
    # -----------------------------------------------------------

    top1_category = top1_model.predict(
        model_input
    )[0]

    top1_probabilities = top1_model.predict_proba(
        model_input
    )[0]

    top1_classes = top1_model.named_steps[
        "classifier"
    ].classes_

    top1_probability = dict(
        zip(
            top1_classes,
            top1_probabilities,
        )
    )[top1_category]

    # -----------------------------------------------------------
    # Ranking model
    # -----------------------------------------------------------

    ranking_probabilities = ranking_model.predict_proba(
        model_input
    )[0]

    ranking_classes = ranking_model.named_steps[
        "classifier"
    ].classes_

    ranked_results = sorted(
        zip(
            ranking_classes,
            ranking_probabilities,
        ),
        key=lambda item: item[1],
        reverse=True,
    )

    return {
        "top_1": {
            "category": top1_category,
            "probability": round(
                float(top1_probability),
                4,
            ),
        },
        "top_2": [
            {
                "category": category,
                "probability": round(
                    float(probability),
                    4,
                ),
            }
            for category, probability in ranked_results[:2]
        ],
        "top_3": [
            {
                "category": category,
                "probability": round(
                    float(probability),
                    4,
                ),
            }
            for category, probability in ranked_results[:3]
        ],
    }


print("INFERENCE MODELS LOADED")
print("-" * 60)
print("Top-1 model loaded successfully.")
print("Ranking model loaded successfully.")

INFERENCE MODELS LOADED
------------------------------------------------------------
Top-1 model loaded successfully.
Ranking model loaded successfully.


## Recreate Holdout Test Split

The holdout partition is read from the shared `split_assignment.csv` created by the modelling notebook, so that inference is tested on the same untouched holdout customers and properties used during model evaluation.

In [2]:
# -------------------------------------------------------------------
# Recreate Holdout Test Split
# -------------------------------------------------------------------

DATA_PATH = Path("../../../../data/processed/hypothesis_testing/regions_anonymised/model_state.csv")
SPLIT_PATH = Path("../../../../data/processed/hypothesis_testing/split_assignment.csv")

model_state = pd.read_csv(
    DATA_PATH,
    parse_dates=["Prediction_Time"],
)

# Separate predictors and target.
X = model_state.drop(
    columns=[
        "Next_Category_ID",
        "Next_Category",
    ]
).copy()

y = model_state["Next_Category"].copy()

# Feature set: pure record. Only the documented-inventory predictors are kept.
RECORD_FEATURES = [
    "documented_asset_count",
    "documented_category_count",
    "documented_value_total",
    "documented_replacement_cost_total",
    "mean_evidence_rating",
    "documented_count_CAT-001",
    "documented_count_CAT-002",
    "documented_count_CAT-003",
    "documented_count_CAT-004",
    "documented_count_CAT-005",
    "documented_count_CAT-006",
    "documented_count_CAT-007",
]

DROPPED_FEATURES = [
    column
    for column in X.columns
    if column not in RECORD_FEATURES
    and column not in ["Property_ID", "Customer_ID", "Prediction_Time"]
]

EXPECTED_PREDICTOR_COUNT = 12

assert len(DROPPED_FEATURES) == 43, (
    f"Expected 43 removed predictors, found {len(DROPPED_FEATURES)}"
)

# Select the holdout rows using the saved partition of each property.
split_assignment = pd.read_csv(SPLIT_PATH)

partition_map = split_assignment.set_index("Property_ID")["Partition"]

partition = X["Property_ID"].map(partition_map)

if partition.isna().any():
    raise ValueError("Some properties have no partition in the split file.")

X_test = X[partition == "test"].copy()
y_test = y.loc[X_test.index].copy()

print("TEST SPLIT RECREATED")
print("-" * 60)

print(f"Test rows:       {len(X_test):,}")
print(f"Test properties: {X_test['Property_ID'].nunique():,}")
print(f"Test customers:  {X_test['Customer_ID'].nunique():,}")


TEST SPLIT RECREATED
------------------------------------------------------------
Test rows:       59
Test properties: 35
Test customers:  30


## End-To-End Inference Test

A real unseen test observation is passed through the saved inference pipeline to verify that the persisted models produce ranked recommendations correctly.

In [3]:
# -------------------------------------------------------------------
# End-To-End Inference Test
# -------------------------------------------------------------------

# Select one unseen test observation.
sample_index = X_test.index[0]

sample_features = X_test.loc[
    [sample_index]
].copy()

actual_category = y_test.loc[
    sample_index
]

# Generate ranked recommendations.
prediction_result = predict_next_categories(
    sample_features
)

print("END-TO-END INFERENCE TEST")
print("-" * 70)

print(f"Property ID:     {sample_features['Property_ID'].iloc[0]}")
print(f"Actual category: {actual_category}")

print()

print("TOP-1 RECOMMENDATION")
print(
    f"{prediction_result['top_1']['category']} "
    f"({prediction_result['top_1']['probability']:.2%})"
)

print()

print("TOP-2 RANKING")
for rank, item in enumerate(
    prediction_result["top_2"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

print()

print("TOP-3 RANKING")
for rank, item in enumerate(
    prediction_result["top_3"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

END-TO-END INFERENCE TEST
----------------------------------------------------------------------
Property ID:     PRP-000029
Actual category: Other Items

TOP-1 RECOMMENDATION
Electronics (83.85%)

TOP-2 RANKING
1. Electronics (79.93%)
2. Other Items (13.61%)

TOP-3 RANKING
1. Electronics (79.93%)
2. Other Items (13.61%)
3. Appliances (2.47%)


## Saved Model Reproduction Check

The saved Top-1 and ranking models are loaded from disk and scored on the holdout partition. The resulting metrics must equal the holdout results written by the modelling notebook, confirming that the persisted models are the ones that produced the reported results.

In [4]:
# -------------------------------------------------------------------
# Saved Model Reproduction Check
# -------------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    top_k_accuracy_score,
)

RESULTS_PATH = Path(
    "../../../../data/processed/hypothesis_testing/regions_anonymised/03_pure_record/results_holdout_test.csv"
)

saved_results = pd.read_csv(RESULTS_PATH).iloc[0]

X_test_model = X_test.drop(
    columns=[
        "Property_ID",
        "Customer_ID",
        "Prediction_Time",
    ] + DROPPED_FEATURES,
    errors="ignore",
).copy()

assert X_test_model.shape[1] == EXPECTED_PREDICTOR_COUNT, (
    f"Expected {EXPECTED_PREDICTOR_COUNT} predictors, "
    f"found {X_test_model.shape[1]}"
)

top1_predictions = top1_model.predict(X_test_model)

ranking_probabilities = ranking_model.predict_proba(X_test_model)

ranking_classes = ranking_model.named_steps["classifier"].classes_

reproduced = {
    "Top1_Accuracy": accuracy_score(y_test, top1_predictions),
    "Top1_Macro_F1": f1_score(
        y_test,
        top1_predictions,
        average="macro",
        zero_division=0,
    ),
    "Top1_Weighted_F1": f1_score(
        y_test,
        top1_predictions,
        average="weighted",
        zero_division=0,
    ),
    "Top2_Accuracy": top_k_accuracy_score(
        y_test,
        ranking_probabilities,
        k=2,
        labels=ranking_classes,
    ),
    "Top3_Accuracy": top_k_accuracy_score(
        y_test,
        ranking_probabilities,
        k=3,
        labels=ranking_classes,
    ),
}

print("SAVED MODEL REPRODUCTION CHECK")
print("-" * 70)

for metric, value in reproduced.items():
    difference = abs(value - saved_results[metric])
    print(
        f"{metric:<18} reproduced={value:.4f} | "
        f"saved={saved_results[metric]:.4f} | "
        f"difference={difference:.2e}"
    )

    assert difference < 1e-9, f"{metric} does not match the saved holdout results."

print()
print("All holdout metrics reproduced from the saved models.")

SAVED MODEL REPRODUCTION CHECK
----------------------------------------------------------------------
Top1_Accuracy      reproduced=0.4576 | saved=0.4576 | difference=0.00e+00
Top1_Macro_F1      reproduced=0.3452 | saved=0.3452 | difference=0.00e+00
Top1_Weighted_F1   reproduced=0.4078 | saved=0.4078 | difference=0.00e+00
Top2_Accuracy      reproduced=0.6949 | saved=0.6949 | difference=0.00e+00
Top3_Accuracy      reproduced=0.8814 | saved=0.8814 | difference=0.00e+00

All holdout metrics reproduced from the saved models.


## Final Conclusion

The saved Top-1 and ranking models of this experiment cell (regions release, pure-record feature set) were loaded from disk and scored on the holdout partition. All five holdout metrics (Top-1 accuracy, Top-1 Macro F1, Top-1 Weighted F1, Top-2 and Top-3 accuracy) match the values written by the modelling notebook to within 6e-17, so the persisted models are the models that produced the reported results. The inference function returns Top-1, Top-2 and Top-3 recommendations for an unseen holdout observation.

Conclusions about the cost of anonymisation and about the contribution of the AI features are drawn from the comparison of all experiment cells, not from this notebook alone.

### Final Deliverables

This run produced:

- a Top-1 and a ranking model persisted in `models/hypothesis_testing/`;
- a reusable inference function that returns Top-1, Top-2 and Top-3 category recommendations;
- a reproduction check linking the saved models to the saved holdout results.